# GraphRAG SDK -- Interactive Demo

This notebook walks through the core GraphRAG SDK workflow:
1. Install the SDK
2. Configure LLM and embedder providers
3. Ingest text into a knowledge graph
4. Query with retrieve() and completion()
5. Multi-turn conversations

**Prerequisites:** FalkorDB running locally (`docker run -p 6379:6379 falkordb/falkordb`)

In [ ]:
# Install the SDK (uncomment if running in Colab or fresh environment)
# !pip install graphrag-sdk[litellm]

## 1. Configure Providers

Set your API key as an environment variable before running:

**OpenAI:** `export OPENAI_API_KEY="sk-..."`

**Azure OpenAI:** See the commented-out section below.

In [ ]:
from graphrag_sdk import GraphRAG, ConnectionConfig, LiteLLM, LiteLLMEmbedder

# Option A: OpenAI (default)
llm = LiteLLM(model="openai/gpt-4o")
embedder = LiteLLMEmbedder(model="openai/text-embedding-3-small")

# Option B: Azure OpenAI (uncomment and fill in)
# import os
# llm = LiteLLM(
#     model="azure/gpt-4.1",
#     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
#     api_base=os.getenv("AZURE_OPENAI_ENDPOINT"),
#     api_version="2024-12-01-preview",
# )
# embedder = LiteLLMEmbedder(
#     model="azure/text-embedding-ada-002",
#     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
#     api_base=os.getenv("AZURE_OPENAI_ENDPOINT"),
#     api_version="2024-12-01-preview",
# )

conn = ConnectionConfig(host="localhost", graph_name="notebook_demo")
print("Providers configured.")

## 2. Ingest Text

We'll ingest a short passage and build a knowledge graph from it.

In [ ]:
TEXT = (
    "Alice Johnson is a software engineer at Acme Corp in London. "
    "She leads the backend team and reports to Bob Smith, the CTO. "
    "Acme Corp was founded in 2015 and specializes in cloud infrastructure. "
    "The company recently opened a new office in Berlin, managed by Clara Wei. "
    "Clara previously worked at TechStart in Munich before joining Acme Corp."
)

rag = GraphRAG(connection=conn, llm=llm, embedder=embedder)

result = await rag.ingest("demo_doc", text=TEXT)
print(f"Nodes created:         {result.nodes_created}")
print(f"Relationships created: {result.relationships_created}")

finalize_result = await rag.finalize()
print(f"\nFinalize complete: {finalize_result}")

## 3. Query the Knowledge Graph

Use `retrieve()` for context-only, or `completion()` for full RAG.

Ask questions and see the answers generated from the graph.

In [ ]:
questions = [
    "Where does Alice work?",
    "Who is the CTO of Acme Corp?",
    "Where did Clara Wei work before Acme Corp?",
]

for q in questions:
    answer = await rag.completion(q)
    print(f"Q: {q}")
    print(f"A: {answer.answer}\n")

## 4. Inspect Provenance

Every answer traces back through entities, relationships, and chunks to the source document. Use `return_context=True` to inspect what was retrieved.

In [ ]:
result = await rag.completion("Where does Alice work?", return_context=True)

print("Answer:", result.answer)
print("\n--- Retrieved Context ---")
if result.retriever_result and result.retriever_result.items:
    for i, item in enumerate(result.retriever_result.items[:5], 1):
        print(f"\n[{i}] Score: {item.score:.3f}")
        print(f"    Text: {item.text[:200]}..." if len(item.text) > 200 else f"    Text: {item.text}")
else:
    print("No retrieval context available.")

## 5. Cleanup

Close the connection when done.

In [ ]:
await rag._conn.close()
print("Connection closed.")